# PyCAM-SIMA dynamic persistent model pool

This Notebook starts one dynamically sized MPI world and splits it into reusable model slots. Forked models copy rank-local StatePool data directly inside that MPI world: no child `qsub`, no child `mpiexec`, and no checkpoint file is required.

## 1. Execution model

```text
Jupyter → Dask Actor → one mpiexec
                         │
                         ├── slot 0: base
                         ├── slot 1: child
                         └── slot N: child / idle

base.fork(...) → matching MPI ranks copy Python-owned StatePool arrays
```

`ranks_per_model` defaults to `ModelConfig.mpi_size`. The number of slots is calculated from the available PBS CPU and memory resources; neither value is hard-coded.

## 2. Configure Dask and inspect the resource plan

In [1]:
from datetime import datetime
from pathlib import Path
import os
import shutil

import numpy as np
import pycam_sima
from dask.distributed import Client
from pycam_sima import DaskExperimentClient

repo = Path('/glade/work/ruitong/pycam-sima')
scratch = Path(os.environ.get('SCRATCH', '/glade/derecho/scratch/ruitong'))
config_path = repo / 'configs/fkessler_model.yaml'
reference_atm_in = repo / 'reference/cases/FKESSLER_ne3pg3_gnu_24x50/CaseDocs/atm_in'
stamp = datetime.now().strftime('%Y%m%d-%H%M%S')
experiment_root = scratch / 'pycam-sima/persistent_pool_trials' / stamp
initial_run_dir = experiment_root / 'initial-run'
initial_run_dir.mkdir(parents=True, exist_ok=False)
shutil.copy2(reference_atm_in, initial_run_dir / 'atm_in')

# Run this Notebook inside a PBS allocation for a local, single-allocation pool.
execution_mode = 'allocation' if os.environ.get('PBS_JOBID') else 'pbs'
client = Client(processes=False, n_workers=1, threads_per_worker=1, dashboard_address=None)
print('pycam_sima version:', pycam_sima.__version__)
print('pycam_sima loaded from:', Path(pycam_sima.__file__).resolve())

experiments = DaskExperimentClient(
    client,
    config=config_path,
    initial_run_dir=initial_run_dir,
    run_root=experiment_root / 'models',
    python_executable=repo / '.venv/bin/python',
    execution_mode=execution_mode,
)

# None inherits mpi_size from the model config. Change it to an integer or 'auto'.
resource_plan = experiments.plan_pool(
    max_concurrent_models=4,
    ranks_per_model=None,
    memory_per_model='auto',
)
resource_plan.describe()

pycam_sima version: 0.16.0
pycam_sima loaded from: /glade/work/ruitong/pycam-sima/src/pycam_sima/__init__.py


{'available_nodes': 4,
 'available_cpus': 96,
 'available_memory_bytes': 343597383680,
 'ranks_per_model': 24,
 'model_slots': 4,
 'world_size': 96,
 'slot_placements': ((0,
   1,
   2,
   3,
   4,
   5,
   6,
   7,
   8,
   9,
   10,
   11,
   12,
   13,
   14,
   15,
   16,
   17,
   18,
   19,
   20,
   21,
   22,
   23),
  (24,
   25,
   26,
   27,
   28,
   29,
   30,
   31,
   32,
   33,
   34,
   35,
   36,
   37,
   38,
   39,
   40,
   41,
   42,
   43,
   44,
   45,
   46,
   47),
  (48,
   49,
   50,
   51,
   52,
   53,
   54,
   55,
   56,
   57,
   58,
   59,
   60,
   61,
   62,
   63,
   64,
   65,
   66,
   67,
   68,
   69,
   70,
   71),
  (72,
   73,
   74,
   75,
   76,
   77,
   78,
   79,
   80,
   81,
   82,
   83,
   84,
   85,
   86,
   87,
   88,
   89,
   90,
   91,
   92,
   93,
   94,
   95)),
 'estimated_model_bytes': 39426808,
 'memory_per_model_bytes': 39426808,
 'reserve_bytes': 51539607552,
 'cpus_per_node': 24,
 'memory_per_node_bytes': 85899345920,


PyCAM-SIMA pool submitted as 6908381.desched1; waiting for 4 x 24 MPI ranks ...


## 3. One pool, one base, and in-memory branches

This is the only pool creation in the Notebook. Closing a child returns its slot; closing the outer `with` stops the one MPI world.

In [2]:
try:
    with experiments.pool(
        name='cam-pool',
        resource_plan=resource_plan,
    ) as pool:
        with pool.model('base') as base:
            temperature_before = base.fields.air_temperature.stats(rank=0)
            base.advance(steps=2)

            # Dynamically add a Python-owned StatePool variable before forking.
            base.fields.create(
                'experiment_tracer',
                dims=('column', 'level'),
                units='kg kg-1',
                initial=0.0,
            )
            # Safe deletion is also collective. Only an unused dynamic
            # field can be removed; model/plugin-required fields fail fast.
            base.fields.create('temporary_probe', dims=('column',), initial=1.0)
            deleted_probe = base.fields.delete('temporary_probe')

            # Dynamically build and install a new original-Fortran
            # calculation function into this live pool model. No MPI
            # restart or StatePool reinitialization is required.
            installed_plugin = base.physics.install(
                source=(
                    repo
                    / 'examples/plugins/runtime_temperature_offset/device.yaml'
                ),
                project_root=repo,
                after='kessler',
                inputs={
                    'runtime_plugin_temperature': 240.0,
                    'runtime_plugin_temperature_increment': 1.5,
                },
            )
            # Current releases return the plugin metadata directly. The
            # fallback keeps this Notebook readable if a running kernel
            # still holds the pre-normalization API in memory.
            if 'name' in installed_plugin:
                installed_plugin_metadata = installed_plugin
            else:
                installed_plugin_metadata = installed_plugin.get(
                    'installed_plugin', {}
                )
            if 'name' not in installed_plugin_metadata:
                raise RuntimeError(
                    'physics.install() returned no plugin name: '
                    f'{installed_plugin!r}'
                )

            plugin_temperature = (
                base.fields.ccpp_runtime_plugin_temperature
            )
            plugin_before = plugin_temperature.stats(rank=0)
            base.physics.scheme(
                'runtime_temperature_offset', group='before'
            ).run()
            plugin_after = plugin_temperature.stats(rank=0)
            assert np.isclose(
                plugin_after['mean'] - plugin_before['mean'], 1.5
            )

            # Each child receives a private, bitwise copy of the base arrays.
            with base.fork('control', 'no_kessler', 'warm') as branches:
                branches.no_kessler.physics.kessler.enabled = False
                branches.warm.fields.air_temperature += 1.0

                control_initial = branches.control.fields.air_temperature.get(rank=0)
                warm_initial = branches.warm.fields.air_temperature.get(rank=0)
                assert np.array_equal(warm_initial, np.add(control_initial, 1.0))

                # The dynamically installed function and its fields are
                # part of the forked model state as well.
                child_plugin_before = (
                    branches.control.fields
                    .ccpp_runtime_plugin_temperature.stats(rank=0)
                )
                branches.control.physics.scheme(
                    'runtime_temperature_offset', group='before'
                ).run()
                child_plugin_after = (
                    branches.control.fields
                    .ccpp_runtime_plugin_temperature.stats(rank=0)
                )
                assert np.isclose(
                    child_plugin_after['mean']
                    - child_plugin_before['mean'],
                    1.5,
                )

                # Run one scheme on one branch only. This executes Kessler
                # without running a complete step or advancing model time.
                control_step_before_scheme = branches.control.status.step
                branches.control.physics.scheme(
                    'kessler', group='before'
                ).run()
                assert (
                    branches.control.status.step
                    == control_step_before_scheme
                )

                # Advance only the control branch by one complete step.
                branch_steps_before_single_advance = {
                    name: status.step
                    for name, status in branches.statuses.items()
                }
                branches.control.advance(steps=1)
                branch_steps_after_single_advance = {
                    name: status.step
                    for name, status in branches.statuses.items()
                }
                assert branch_steps_after_single_advance['control'] == (
                    branch_steps_before_single_advance['control'] + 1
                )
                assert branch_steps_after_single_advance['no_kessler'] == (
                    branch_steps_before_single_advance['no_kessler']
                )
                assert branch_steps_after_single_advance['warm'] == (
                    branch_steps_before_single_advance['warm']
                )

                # Advance every occupied child branch by one complete step.
                branches.advance(steps=1)
                branch_statuses = branches.statuses

            pool_status = pool.status
            final_base_status = base.status

        result = {
            'resource_plan': resource_plan.describe(),
            'pool_mpi_launch_count': pool_status['mpi_launch_count'],
            'base_step': final_base_status.step,
            'temperature_before': temperature_before,
            'installed_plugin': installed_plugin_metadata['name'],
            'plugin_temperature_before': plugin_before,
            'plugin_temperature_after': plugin_after,
            'child_plugin_temperature_before': child_plugin_before,
            'child_plugin_temperature_after': child_plugin_after,
            'branch_steps_after_single_advance': (
                branch_steps_after_single_advance
            ),
            'branch_steps_final': {
                name: status.step
                for name, status in branch_statuses.items()
            },
            'slots_after_children_close': pool.slots,
        }
finally:
    client.close()

result

{'resource_plan': {'available_nodes': 4,
  'available_cpus': 96,
  'available_memory_bytes': 343597383680,
  'ranks_per_model': 24,
  'model_slots': 4,
  'world_size': 96,
  'slot_placements': ((0,
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    10,
    11,
    12,
    13,
    14,
    15,
    16,
    17,
    18,
    19,
    20,
    21,
    22,
    23),
   (24,
    25,
    26,
    27,
    28,
    29,
    30,
    31,
    32,
    33,
    34,
    35,
    36,
    37,
    38,
    39,
    40,
    41,
    42,
    43,
    44,
    45,
    46,
    47),
   (48,
    49,
    50,
    51,
    52,
    53,
    54,
    55,
    56,
    57,
    58,
    59,
    60,
    61,
    62,
    63,
    64,
    65,
    66,
    67,
    68,
    69,
    70,
    71),
   (72,
    73,
    74,
    75,
    76,
    77,
    78,
    79,
    80,
    81,
    82,
    83,
    84,
    85,
    86,
    87,
    88,
    89,
    90,
    91,
    92,
    93,
    94,
    95)),
  'estimated_model_bytes': 39426808,
  'memory

## 4. Legacy compatibility: one Actor per MPI model

This optional cell preserves the complete pre-pool demonstration: persistent steps, dynamic variables, runtime Fortran plugins, direct scheme/phase calls, checkpoints, and the old multi-Actor fork. It is not the recommended fork path because every legacy child owns a separate MPI launch. Leave both flags `False` during the normal pool demonstration.

In [ ]:
run_legacy_compatibility = False
run_legacy_fork = False

if not run_legacy_compatibility:
    legacy_result = 'skipped; set run_legacy_compatibility = True to run'
else:
    # The complete pre-pool Persistent Dask demonstration is retained
    # here for compatibility and comparison.
    legacy_client = Client(
        processes=False,
        n_workers=(3 if run_legacy_fork else 1),
        threads_per_worker=1,
        dashboard_address=None,
    )
    legacy_experiments = DaskExperimentClient(
        legacy_client,
        config=config_path,
        initial_run_dir=initial_run_dir,
        run_root=experiment_root / 'legacy-models',
        python_executable=repo / '.venv/bin/python',
        execution_mode=execution_mode,
    )
    try:
        # A. Start one PersistentCAMActor and reuse the same MPI model.
        with legacy_experiments.model('legacy-base') as legacy_model:
            legacy_started = legacy_model.status
            legacy_temperature_before = (
                legacy_model.fields.air_temperature.stats(rank=0)
            )
            legacy_model.advance(steps=2)
            legacy_temperature_after = (
                legacy_model.fields.air_temperature.stats(rank=0)
            )
            legacy_checkpoint = legacy_model.save()
            legacy_after_two_steps = legacy_model.status

            assert legacy_started.mpi_launch_count == 1
            assert legacy_after_two_steps.mpi_launch_count == 1
            assert legacy_after_two_steps.worker_pid == legacy_started.worker_pid
            assert legacy_after_two_steps.step == legacy_started.step + 2

            # B. Dynamically add a Python-owned StatePool variable.
            legacy_model.fields.create(
                'legacy_experiment_tracer',
                dims=('column', 'level'),
                units='kg kg-1',
                initial=0.0,
            )
            legacy_tracer = (
                legacy_model.fields.legacy_experiment_tracer.stats(rank=0)
            )
            assert legacy_tracer['mean'] == 0.0

            # C. Build, load, place, and call a runtime Fortran plugin.
            legacy_installed = legacy_model.physics.install(
                source=(
                    repo
                    / 'examples/plugins/runtime_temperature_offset/device.yaml'
                ),
                project_root=repo,
                after='kessler',
                inputs={
                    'runtime_plugin_temperature': 240.0,
                    'runtime_plugin_temperature_increment': 1.5,
                },
            )
            if 'name' in legacy_installed:
                legacy_installed_metadata = legacy_installed
            else:
                legacy_installed_metadata = legacy_installed.get(
                    'installed_plugin', {}
                )
            if 'name' not in legacy_installed_metadata:
                raise RuntimeError(
                    'physics.install() returned no plugin name: '
                    f'{legacy_installed!r}'
                )
            legacy_plugin_temperature = (
                legacy_model.fields.ccpp_runtime_plugin_temperature
            )
            legacy_plugin_before = legacy_plugin_temperature.stats(rank=0)
            legacy_model.physics.scheme(
                'runtime_temperature_offset', group='before'
            ).run()
            legacy_plugin_after = legacy_plugin_temperature.stats(rank=0)
            assert np.isclose(
                legacy_plugin_after['mean']
                - legacy_plugin_before['mean'],
                1.5,
            )

            # D. Explicit scheme and phase calls do not advance time;
            # a complete model step does.
            legacy_before_control = legacy_model.status
            legacy_model.physics.kessler.run()
            legacy_after_scheme = legacy_model.status
            legacy_model.phases.physics_to_dynamics.run()
            legacy_after_phase = legacy_model.status
            legacy_model.advance(steps=1)
            legacy_after_step = legacy_model.status

            assert legacy_after_scheme.step == legacy_before_control.step
            assert legacy_after_phase.step == legacy_before_control.step
            assert legacy_after_step.step == legacy_before_control.step + 1

            # E. Optional old-style fork. Unlike the pool path, each child
            # starts a separate MPI model and receives a CheckpointBundle.
            if not run_legacy_fork:
                legacy_fork_result = (
                    'skipped; set run_legacy_fork = True to run'
                )
            elif execution_mode != 'pbs':
                legacy_fork_result = (
                    'skipped; legacy fork requires PBS mode'
                )
            else:
                legacy_control = legacy_experiments.plan('legacy-control')
                legacy_no_kessler = legacy_experiments.plan(
                    'legacy-no-kessler', experimental=True
                )
                legacy_no_kessler.physics.kessler.disable()
                legacy_warm = legacy_experiments.plan('legacy-warm')
                legacy_warm.fields.edit(
                    'air_temperature', 'add', 1.0
                )
                legacy_base_temperature = (
                    legacy_model.fields.air_temperature.get(rank=0)
                )
                legacy_children = legacy_experiments.fork_models(
                    legacy_model,
                    (
                        legacy_control,
                        legacy_no_kessler,
                        legacy_warm,
                    ),
                    close_parent=False,
                )
                with legacy_children:
                    legacy_initial_temperature = {
                        name: child.fields.air_temperature.get(rank=0)
                        for name, child in legacy_children.items()
                    }
                    assert np.array_equal(
                        legacy_initial_temperature['legacy-control'],
                        legacy_base_temperature,
                    )
                    assert np.array_equal(
                        legacy_initial_temperature['legacy-no-kessler'],
                        legacy_base_temperature,
                    )
                    assert np.array_equal(
                        legacy_initial_temperature['legacy-warm'],
                        np.add(legacy_base_temperature, 1.0),
                    )
                    legacy_children.advance(steps=1)
                    legacy_fork_result = legacy_children.statuses

            legacy_finished = legacy_model.status
            legacy_result = {
                'model': legacy_finished.name,
                'pbs_job_id': legacy_finished.pbs_job_id,
                'worker_pid': legacy_finished.worker_pid,
                'mpi_launch_count': legacy_finished.mpi_launch_count,
                'start_step': legacy_started.step,
                'final_step': legacy_finished.step,
                'temperature_before': legacy_temperature_before,
                'temperature_after_two_steps': legacy_temperature_after,
                'dynamic_variable': legacy_tracer,
                'installed_plugin': legacy_installed_metadata['name'],
                'plugin_temperature_before': legacy_plugin_before,
                'plugin_temperature_after': legacy_plugin_after,
                'checkpoint': str(legacy_checkpoint.path),
                'fork': legacy_fork_result,
            }
    finally:
        legacy_client.close()

legacy_result